In [0]:
from pyspark.sql.functions import col, regexp_replace, to_timestamp, current_timestamp, coalesce, count, when

# 1. Leer la tabla directamente desde la Capa Bronze
df_bronze = spark.read.table("workspace.default.bronze_mining")

# 2. Identificar columnas numéricas
metadata_cols = ["date", "ingestion_timestamp", "source_file"]
numeric_cols = [c for c in df_bronze.columns if c not in metadata_cols]

# -------------------------------------------------------------------
# PASO A: Normalización de Formato (Casting & Trim)
# -------------------------------------------------------------------
df_silver = df_bronze

# Reemplazar comas por puntos y castear a Double (Transformación del dato)
for c in numeric_cols:
    df_silver = df_silver.withColumn(c, regexp_replace(col(c), ",", ".").cast("double"))

# Convertir columna 'date' a Timestamp real (Formato en la Base de Datos)
df_silver = df_silver.withColumn(
    "timestamp", 
    coalesce(
        to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("date"), "yyyy/MM/dd HH:mm:ss"),
        to_timestamp(col("date"), "dd/MM/yyyy HH:mm:ss"),
        to_timestamp(col("date"), "dd/MM/yyyy H:mm")
    )
).drop("date")

# -------------------------------------------------------------------
# PASO B: Reglas de Calidad e Integridad de Datos (Corregido)
# -------------------------------------------------------------------

# 1. Eliminar filas sin fecha válida
df_silver = df_silver.filter(col("timestamp").isNotNull())

# 2. Eliminar duplicados REALES (filas donde absolutamente TODAS las columnas sean idénticas)
df_silver = df_silver.dropDuplicates()

# 3. Filtrar límites físicos de las variables principales (0% al 100%)
df_silver = df_silver.filter(
    (col("pct_Iron_Feed") >= 0) & (col("pct_Iron_Feed") <= 100) &
    (col("pct_Silica_Feed") >= 0) & (col("pct_Silica_Feed") <= 100)
)

# -------------------------------------------------------------------
# PASO C: Metadatos de Auditoría de Silver
# -------------------------------------------------------------------
df_silver = df_silver.withColumn("silver_processed_at", current_timestamp())

# Muestra los resultados finales
print("Limpieza completada exitosamente.")
display(df_silver.limit(5))

filas_bronze = df_bronze.count()
filas_silver = df_silver.count()
eliminados = filas_bronze - filas_silver

print(f"Filas originales (Bronze): {filas_bronze:,}")
print(f"Filas limpias (Silver):     {filas_silver:,}")
print(f"Filas/Duplicados filtrados: {eliminados:,}")

In [0]:
table_silver_name = "workspace.default.silver_mining_quality"

(df_silver.write
 .format("delta")
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable(table_silver_name))

print(f"Tabla {table_silver_name} guardada con éxito en la Capa Silver.")